# 75 — Render the overlay A-Box

Six qualified concepts and eight recordings, from the two authored instance files
in `overlay/`. Runs after `70_generate_qbc.ipynb` (it resolves enum values out of
both T-Boxes) and after `50_render_bc.ipynb` (it checks its own join against the
core A-Box).

The decisions this exercises, and what each does to the graph:

| | |
|---|---|
| **D13** | subjects are `https://w3id.org/cdisc/cosmos/qbc/{Name}` |
| **D14** | the analyte link is `skos:broader`, **not** `broadMatch` — see the note below |
| **D15** | interpretation-regime assertions render with no `regime` pointer |
| **D16** | authored mappings become real SKOS predicates, in deliberate contrast to D10 |
| **D17** | a recording's subject **is** the D3 Dataset Specialization IRI |
| **D20** | result scales resolve to the **core** enum IRIs, so the overlay shares those nodes |
| **D21** | a data element concept's `dataType` hangs on the sibling's use of it, never on the shared NCIt node |
| **D22** | the overlay writes only onto its own nodes: an admissible specimen is the sibling's *use* of an NCIt concept, a missing value set is absent rather than a `[VERIFY]` value, and a recording's specimen is reported against the admissible set, not asserted |

**Why `skos:broader` and not `skos:broadMatch`.** D14 first settled on
`broadMatch`, on the argument that SKOS makes it a sub-property of `broader` so
the weaker statement follows by entailment. Rendering D16 alongside it showed why
that does not work here: HCV RNA maps to LOINC `111469-3` with relation
`broadMatch`, so one subject carried eight `skos:broadMatch` triples — six analyte
links and two external mappings — distinguishable only by the target's namespace.
`broader` for this repo's own concept hierarchy, `*Match` for mappings out to
external code systems, keeps the two apart.

## Configuration

In [ ]:
DOWNLOADS = "../downloads"
OVERLAY   = "../overlay"
ROOT      = ".."

SOURCES = ["glucose.instances.yaml", "hcvrna.instances.yaml"]

QBC_TBOX  = f"{ROOT}/cosmos_qbc_v1.ttl"
CORE_TBOX = f"{ROOT}/cosmos_bc_v1.ttl"
CORE_ABOX = f"{ROOT}/cosmos_bc_v1.instances.ttl"
TARGET    = "cosmos_qbc_v1.instances.ttl"

QBC_NS = "https://w3id.org/cdisc/cosmos/qbc/"
BC_NS  = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"
OBO    = "http://purl.obolibrary.org/obo/NCIT_"
DSS_NS = "https://w3id.org/cdisc/cosmos/dss/"

VERSION      = "0.2.0"
ONTOLOGY_IRI = QBC_NS
CORE_BC_INSTANCES_IRI = "https://w3id.org/cdisc/cosmos/bc/instances/"

## Identity rules, as functions

`core_enum` and `qbc_enum` are the same guard `50_render_bc.ipynb` uses: a
permissible value is looked up in the T-Box that declares it, and a value the
T-Box does not have raises rather than being written.

The split between them is decision D20. Result scales and data types are CDISC's
enums, so their permissible values come from the **core** T-Box and the overlay
shares those nodes with `cosmos_bc_v1.instances.ttl`. `MappingRelationEnum` is the
overlay's own, so it comes from the overlay T-Box.

In [ ]:
from pathlib import Path

import yaml
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, XSD

QBC = Namespace(QBC_NS)
COSMOS_BC = Namespace(BC_NS)

core_tbox = Graph().parse(CORE_TBOX, format="turtle")
qbc_tbox = Graph().parse(QBC_TBOX, format="turtle")

# D16: the authored mapping relation becomes the SKOS predicate it names.
# Contrast D10, which refuses to assert any relation for a published COSMoS
# coding. The difference is that these were curated one at a time against the
# LOINC service and are reviewable; those were not.
MAPPING_PREDICATE = {
    "exactMatch": SKOS.exactMatch,
    "narrowMatch": SKOS.narrowMatch,
    "broadMatch": SKOS.broadMatch,
}


def core_enum(enum_name, value):
    """Permissible-value IRI as the CORE T-Box declares it (D20)."""
    iri = URIRef(f"{BC_NS}{enum_name}#{value}")
    if (iri, RDF.type, OWL.Class) not in core_tbox:
        raise RuntimeError(f"{value!r} is not a permissible value of core {enum_name}")
    return iri


def qbc_enum(enum_name, value):
    """Permissible-value IRI as the OVERLAY T-Box declares it."""
    iri = URIRef(f"{QBC_NS}{enum_name}#{value}")
    if (iri, RDF.type, OWL.Class) not in qbc_tbox:
        raise RuntimeError(f"{value!r} is not a permissible value of overlay {enum_name}")
    return iri


def ncit(code):
    return URIRef(OBO + code)


def state_segment(value):
    """A state value in an IRI path. '*' means any state and cannot be a path segment."""
    return "any" if value == "*" else value


sources = {name: yaml.safe_load(Path(OVERLAY, name).read_text(encoding="utf-8")) for name in SOURCES}
for name, data in sources.items():
    print(f"{name:28s} {len(data['siblings'])} qualified concepts  {len(data.get('recordings', []))} recordings")

## The qualified concepts

Two rules are worth reading off the code rather than inferring.

**A data element concept is rendered as the sibling's *use* of it** — a node at
`{sibling}/dec/{code}` carrying `dataType` and `exampleSet`, linked to the shared
NCIt concept by `cosmos_bc:conceptId` rendered as an edge. It is not rendered onto
`obo:NCIT_C70856` itself. That is decision D21, and it is the shape the core
A-Box will move to: measured 2026-09-01, `data_type` and `example_set` are
properties of the (BC, DEC) pair and not of the DEC — `C70856` Observation Result
carries seven different `data_type` values across the pinned package. Attaching
the overlay's claim to the shared node would assert a contradiction onto it.

**The same rule holds for admissible specimens** (decision D22). An admissible
specimen is rendered as the sibling's use of an NCIt concept — a node at
`{sibling}/specimen/{code}` typed `ConceptTerm`, carrying `preferredTerm`, reaching
the shared PURL by `conceptId` as an edge — and nothing is written onto the PURL
itself. Before D22 the overlay typed `obo:NCIT_C13325` as `qbc:ConceptTerm`
directly; no other graph described that node yet, so nothing contradicted, but it
was the move D21 refuses for `dataType`, and the DSS layer will describe specimens.

**A semantic value set is either curated or absent** (D22). Two siblings carried a
term whose value was the literal `[VERIFY]` — a placeholder in the value position,
so a consumer asking what the observation can say got the marker back. Decision D15
already settled the shape for this: source and deliverable agree, and nothing
invented exists anywhere. The two entries are gone from the instance files, the
YAML comment explaining the absence stays, and the renderer refuses any value that
starts with `[VERIFY]`. `sourceAnchor` keeps the convention; it is provenance, not
a value.

**Both the SKOS predicate and the schema's own slot are emitted** for the analyte
link. `skos:broader` carries the meaning to any consumer; `qbc:broaderConceptId`
is the schema's slot rendered as an edge, exactly as the core A-Box renders
`parentConceptId`, and it violates the published shape in exactly the same way
(decision D11, cause 3).

In [ ]:
g = Graph()
counts = {"siblings": 0, "dec_uses": 0, "mappings": 0, "value_terms": 0, "regimes": 0}
specimens = set()

for name, data in sources.items():
    for sibling in data["siblings"]:
        s = URIRef(sibling["iri"])
        counts["siblings"] += 1

        g.add((s, RDF.type, QBC.QualifiedBiomedicalConcept))
        g.add((s, COSMOS_BC.shortName, Literal(sibling["shortName"])))
        g.add((s, COSMOS_BC.definition, Literal(sibling["definition"].strip())))

        broader = ncit(sibling["broaderConceptId"])
        g.add((s, SKOS.broader, broader))          # D14
        g.add((s, QBC.broaderConceptId, broader))  # the schema's slot, as an edge

        g.add((s, QBC.resultScale, core_enum("BiomedicalConceptResultScaleEnum", sibling["resultScale"])))

        for slot, decs in (("resultDataElementConcept", [sibling["resultDataElementConcept"]]),
                           ("qualifierDataElementConcepts", sibling.get("qualifierDataElementConcepts", []))):
            for dec in decs:
                use = URIRef(f"{s}/dec/{dec['conceptId']}")
                counts["dec_uses"] += 1
                g.add((s, QBC[slot], use))
                g.add((use, RDF.type, COSMOS_BC.DataElementConcept))
                g.add((use, COSMOS_BC.conceptId, ncit(dec["conceptId"])))
                g.add((use, COSMOS_BC.shortName, Literal(dec["shortName"])))
                if dec.get("dataType"):
                    g.add((use, COSMOS_BC.dataType,
                           core_enum("DataElementConceptDataTypeEnum", dec["dataType"])))
                for example in dec.get("exampleSet", []):
                    g.add((use, COSMOS_BC.exampleSet, Literal(example)))

        for term in sibling.get("admissibleSpecimens", []):
            use = URIRef(f"{s}/specimen/{term['conceptId']}")
            specimens.add(term["conceptId"])
            g.add((s, QBC.admissibleSpecimens, use))                 # D22
            g.add((use, RDF.type, QBC.ConceptTerm))
            g.add((use, QBC.conceptId, ncit(term["conceptId"])))
            g.add((use, QBC.preferredTerm, Literal(term["preferredTerm"])))

        for mapping in sibling.get("externalMappings", []):
            external = URIRef(mapping["system"] + mapping["code"])
            node = URIRef(f"{s}/mapping/{mapping['code']}")
            counts["mappings"] += 1
            g.add((s, QBC.externalMappings, node))
            g.add((s, MAPPING_PREDICATE[mapping["relation"]], external))
            g.add((node, RDF.type, QBC.ExternalMapping))
            g.add((node, QBC.relation, qbc_enum("MappingRelationEnum", mapping["relation"])))
            g.add((node, QBC.code, Literal(mapping["code"])))
            g.add((node, QBC.system, Literal(mapping["system"])))
            if mapping.get("comment"):
                g.add((node, QBC.comment, Literal(mapping["comment"])))

        for index, term in enumerate(sibling.get("semanticValueSet", []), start=1):
            if term["value"].startswith("[VERIFY]"):
                raise RuntimeError("D22: a placeholder is not a value; leave the slot absent instead")
            node = URIRef(f"{s}/value/{index}")
            counts["value_terms"] += 1
            g.add((s, QBC.semanticValueSet, node))
            g.add((node, RDF.type, QBC.SemanticValueSetTerm))
            g.add((node, QBC.value, Literal(term["value"])))
            if term.get("ncitCode"):
                g.add((node, QBC.ncitCode, ncit(term["ncitCode"])))
            if term.get("sourceAnchor"):
                g.add((node, QBC.sourceAnchor, Literal(term["sourceAnchor"])))

        for assertion in sibling.get("interpretationRegimes", []):
            if assertion.get("regime"):
                raise RuntimeError("D15: the regime slot must stay empty; no governed vocabulary exists")
            state = assertion["stateDataElementConceptId"]
            node = URIRef(f"{s}/regime/{state}/{state_segment(assertion['stateValue'])}")
            counts["regimes"] += 1
            g.add((s, QBC.interpretationRegimes, node))
            g.add((node, RDF.type, QBC.InterpretationRegimeAssertion))
            g.add((node, QBC.stateDataElementConceptId, ncit(state)))
            g.add((node, QBC.stateValue, Literal(assertion["stateValue"])))
            if assertion.get("sourceAnchor"):
                g.add((node, QBC.sourceAnchor, Literal(assertion["sourceAnchor"])))

for label, value in counts.items():
    print(f"{label:26s} {value:>5,}")
print(f"{'admissible specimens':26s} {len(specimens):>5,}  (distinct NCIt concepts)")
print(f"{'triples so far':26s} {len(g):>5,}")

## The recordings — decision D17

**A recording's subject is the Dataset Specialization IRI decision D3 settled**,
`.../cosmos/dss/{DOMAIN}/{MNEMONIC}`. Not a separate node pointing at one.

A recording and a dataset specialization are the same specialization described at
two grains: the overlay says what survives once the qualified concept carries
scale, specimen and result type, and the DSS layer will say the rest. Giving them
one IRI means the two descriptions merge on one node when the DSS A-Box lands,
rather than needing a mapping between them. D4 defers that layer, so all eight
IRIs currently have no other triples — dangling by design, counted in the checks
below rather than hidden.

**A recording's specimen is reported against the sibling's admissible set, not
asserted to be in it** (decision D22). The schema once said it must be; HCV RNA
shows why it cannot be: the admissible set is NCIt concept-level — Serum `C13325`,
Plasma `C13356` — while the SDTM specimen is a CT submission value, and CT carries
the composite `SERUM OR PLASMA` `C105706`. Two grains, and the recording lives at
the SDTM one. So membership is printed per recording, and the count of composites
is a measurement rather than a failure.

`loincPins` stays a literal. `externalMappings` composes an IRI because the
instance states a `system`; `loincPins` states none, so composing one would be an
authored act the slot does not license — the same line decision D10 draws.

In [ ]:
recordings = 0
membership = []

admissible = {sibling["iri"]: {t["conceptId"] for t in sibling.get("admissibleSpecimens", [])}
              for data in sources.values() for sibling in data["siblings"]}

for name, data in sources.items():
    for recording in data.get("recordings", []):
        r = URIRef(f"{DSS_NS}{recording['domain']}/{recording['recordingId']}")
        recordings += 1

        g.add((r, RDF.type, QBC.Recording))
        g.add((r, DCTERMS.identifier, Literal(recording["recordingId"])))
        g.add((r, QBC.siblingIri, URIRef(recording["siblingIri"])))
        g.add((r, QBC.domain, Literal(recording["domain"])))
        g.add((r, QBC.testCode, Literal(recording["testCode"])))

        specimen = recording.get("specimen")
        if specimen:
            node = URIRef(f"{r}/specimen")
            g.add((r, QBC.specimen, node))
            g.add((node, RDF.type, QBC.AssignedTerm))
            g.add((node, QBC.conceptId, ncit(specimen["conceptId"])))
            g.add((node, QBC.value, Literal(specimen["value"])))
            membership.append((recording["recordingId"], specimen["conceptId"], specimen["value"],
                               specimen["conceptId"] in admissible[recording["siblingIri"]]))

        for pin in recording.get("loincPins", []):
            g.add((r, QBC.loincPins, Literal(pin)))
        for slot in ("unitCodelistBinding", "resultCodelistBinding"):
            if recording.get(slot):
                g.add((r, QBC[slot], Literal(recording[slot])))
        for override in recording.get("variableOverrides", []):
            g.add((r, QBC.variableOverrides, Literal(override)))

print(f"{'recordings':26s} {recordings:>5,}")
print(f"{'triples':26s} {len(g):>5,}")
print()
print("specimen against the sibling's admissible set (D22):")
for recording_id, code, value, member in membership:
    print(f"    {recording_id:16s} {code:8s} {value:18s} {'member' if member else 'composite, not a member'}")
print(f"    {sum(1 for m in membership if m[3])} of {len(membership)} recordings use a member of the set")

## Ontology header and canonical write — decision D9

In [ ]:
import json

from rdflib.compare import isomorphic, to_canonical_graph

VANN = Namespace("http://purl.org/vocab/vann/")

INSTANCES_IRI = ONTOLOGY_IRI + "instances/"

meta = json.loads(Path(DOWNLOADS, ".fetch_meta_bc_export.json").read_text(encoding="utf-8"))

ontology = URIRef(INSTANCES_IRI)
g.add((ontology, RDF.type, OWL.Ontology))
g.add((ontology, OWL.imports, URIRef(ONTOLOGY_IRI)))
g.add((ontology, OWL.imports, URIRef(CORE_BC_INSTANCES_IRI)))
g.add((ontology, RDFS.label, Literal("CDISC COSMoS Qualified Biomedical Concepts (instances)")))
g.add((ontology, RDFS.comment, Literal(
    "Authored qualified biomedical concepts and their recordings. Each is linked "
    "to the COSMoS concept it qualifies by skos:broader; that concept is rendered "
    "in cosmos_bc_v1.instances.ttl, which this graph imports. Recordings carry the "
    "Dataset Specialization IRI of decision D3; the Dataset Specialization layer "
    "itself is deferred, so those IRIs have no other triples yet. "
    "Draft - not a normative CDISC artifact.")))
g.add((ontology, DCTERMS.identifier, Literal(meta["package_date"])))
g.add((ontology, OWL.versionIRI, URIRef(INSTANCES_IRI + VERSION)))
g.add((ontology, OWL.versionInfo, Literal(f"v{VERSION}")))

canonical = to_canonical_graph(g)
if not isomorphic(canonical, g) or len(canonical) != len(g):
    raise RuntimeError("canonicalization changed the graph")

out = Graph()
for triple in canonical:
    out.add(triple)
out.bind("qbc", QBC_NS)
out.bind("cosmos_bc", BC_NS)
out.bind("dcterms", DCTERMS)
out.bind("skos", SKOS)
out.bind("NCIT", OBO)

turtle = out.serialize(format="turtle")
Path(ROOT, TARGET).write_text(turtle, encoding="utf-8")
print(f"{TARGET}  {len(out):,} triples  {len(turtle):,} chars")

## Does the overlay actually join the core?

That is the one claim this whole phase rests on, so it is asserted rather than
described. Six checks, all fail-fast.

1. Every `skos:broader` target is a node the core A-Box already types as a
   `BiomedicalConcept`.
2. No `*Match` target is a node the core graph already describes. Together with
   check 1 this is the separation D14 was revised for: `broader` lands inside
   the core graph, a mapping lands outside it. Note that the discriminator has
   to be graph membership rather than the target's host — NCIt PURLs are both
   this repo's identity scheme and a code system the overlay maps out to, so
   `purl.obolibrary.org` appears on both sides.
3. No instance IRI collides with a term IRI in the overlay T-Box. The overlay
   names concepts and classes in one namespace, so this could happen.
4. Every recording is a well-formed D3 Dataset Specialization IRI, and the number
   with no other triples is reported — it should be all of them while D4 defers
   that layer.
5. No `dataType` is asserted on a shared NCIt concept node (D21).
6. No `qbc:regime` triple exists (D15).
7. No NCIt PURL is the subject of any triple in this graph (D22, generalizing 5):
   the overlay reaches shared concepts by edges and writes onto its own nodes only.
8. No `qbc:value` starts with `[VERIFY]` (D22).

In [ ]:
written = Graph().parse(Path(ROOT, TARGET), format="turtle")
core_abox = Graph().parse(CORE_ABOX, format="turtle")
failures = []


def check(name, actual, expected):
    if actual == expected:
        print(f"ok    {name}: {actual}")
    else:
        print(f"FAIL  {name}: expected {expected}, got {actual}")
        failures.append(name)


broader = sorted(written.objects(None, SKOS.broader), key=str)
in_core = [b for b in broader if (b, RDF.type, COSMOS_BC.BiomedicalConcept) in core_abox]
check("skos:broader targets present in the core A-Box", len(in_core), len(broader))
for target in sorted(set(broader), key=str):
    print(f"        {target}  {core_abox.value(target, COSMOS_BC.shortName)}")

# The discriminator is graph membership, not host: NCIt PURLs are this
# repo's own identity scheme (D2) AND a code system the overlay maps out
# to, so the same host appears on both sides. skos:broader must land
# inside the core graph; a *Match must land outside it.
for predicate in (SKOS.exactMatch, SKOS.narrowMatch, SKOS.broadMatch):
    inside = [str(o) for o in written.objects(None, predicate)
              if (o, None, None) in core_abox]
    check(f"{predicate.split('#')[1]} targets outside the core graph", inside, [])

terms = {str(s) for s in qbc_tbox.subjects() if str(s).startswith(QBC_NS)}
subjects = {str(s) for s in written.subjects() if str(s).startswith(QBC_NS)}
check("instance IRIs colliding with a T-Box term", sorted(terms & subjects), [])

recs = sorted(written.subjects(RDF.type, QBC.Recording), key=str)
check("recordings under the D3 Dataset Specialization namespace",
      [str(r) for r in recs if not str(r).startswith(DSS_NS)], [])
dangling = [r for r in recs if (r, None, None) not in core_abox]
print(f"note  recordings whose Dataset Specialization is not rendered (D4): {len(dangling)}/{len(recs)}")

shared = {o for o in written.objects(None, COSMOS_BC.conceptId)}
check("dataType asserted on a shared NCIt concept (D21)",
      [str(c) for c in shared if (c, COSMOS_BC.dataType, None) in written], [])

check("qbc:regime triples (D15)", len(list(written.triples((None, QBC.regime, None)))), 0)

check("NCIt PURLs used as a subject (D22)",
      sorted(str(s) for s in set(written.subjects()) if str(s).startswith(OBO)), [])

check("qbc:value placeholders (D22)",
      sorted(str(o) for o in written.objects(None, QBC.value) if str(o).startswith("[VERIFY]")), [])

if failures:
    raise RuntimeError(f"{len(failures)} check(s) failed: {failures}")
print()
print("the overlay joins the core graph")